In [ ]:
import pandas as pd
from pathlib import Path

In [11]:
MONTH_MAP = {
    'jan': '01', 'feb': '02', 'mar': '03', 'april': '04', 'apr': '04',
    'may': '05', 'jun': '06', 'june': '06', 'july': '07', 'jul': '07',
    'aug': '08', 'sep': '09', 'sept': '09', 'oct': '10', 'nov': '11', 'dec': '12'
}

def parse_report_month(filename):
    """Extract YYYY-MM-01 date from filenames ending in -mon-YYYY.xlsx"""
    name = filename.lower().replace('.xlsx', '')
    parts = name.split('-')
    year = parts[-1]
    month_str = parts[-2]
    month_num = MONTH_MAP.get(month_str)
    if not month_num:
        raise ValueError(f"Could not parse month from filename: {filename}")
    return f"{year}-{month_num}-01"

# Aggregate Monthly Enrollment Data by Group (2014-2019)

In [2]:
enrollment_by_rg = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/monthly-enrollment-by-risk-group.xlsx", sheet_name='Caseload by RG', skiprows=2)

In [3]:
 # the last few rows are blank or contain notes, so we trim to just the data
enrollment_by_rg = enrollment_by_rg[0:138]

# standardize the column names to be lowercase, with underscores instead of spaces, and no special characters. This is necessary for loading into Snowflake.
enrollment_by_rg.columns = (enrollment_by_rg.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('*', '', regex=False)
    .str.replace('&', 'and', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('\u2019', '', regex=False)  # curly apostrophe
    .str.replace("'", '', regex=False)        # straight apostrophe fallback
    .str.replace('.', '_', regex=False)       # handles the .1 suffix
)
# drop total column since it's just the sum of the other columns and can be calculated in Snowflake if needed
enrollment_by_rg = enrollment_by_rg.drop(columns=['childrens_and_chip_total'])

# rename the columns to be more descriptive and clear about the risk groups.
enrollment_by_rg = enrollment_by_rg.rename(columns={
    'childrens_medicaid':   'childrens_medicaid_risk_group',
    'childrens_medicaid_1': 'childrens_medicaid_chip_group',
})

# add a timestamp column to track when the data was loaded into Snowflake
enrollment_by_rg['loaded_at'] = pd.Timestamp.now()

KeyError: "['childrens_and_chip_total'] not found in axis"

In [ ]:
# Check which columns have ANY fractional values
for col in enrollment_by_rg.columns:
    if col not in ['month', 'loaded_at']:
        has_decimal = (enrollment_by_rg[col] % 1 != 0).any()
        first_decimal = enrollment_by_rg[enrollment_by_rg[col] % 1 != 0]['month'].min() if has_decimal else None
        print(f"{col}: fractional={has_decimal}, first_occurrence={first_decimal}")

medicaid_caseload: fractional=True, first_occurrence=2025-08-01 00:00:00
aged_and_medicare_related: fractional=True, first_occurrence=2025-08-01 00:00:00
disability_related: fractional=True, first_occurrence=2025-08-01 00:00:00
parents: fractional=True, first_occurrence=2025-08-01 00:00:00
pregnant_women: fractional=True, first_occurrence=2025-08-01 00:00:00
breast_and_cervical_cancer: fractional=True, first_occurrence=2025-08-01 00:00:00
childrens_medicaid: fractional=True, first_occurrence=2025-08-01 00:00:00
medicaid_clients_under_21: fractional=True, first_occurrence=2025-09-01 00:00:00
medicaid_clients_21_and_older: fractional=True, first_occurrence=2025-08-01 00:00:00
childrens_medicaid_1: fractional=True, first_occurrence=2025-08-01 00:00:00
regular_chip: fractional=False, first_occurrence=None
total: fractional=True, first_occurrence=2025-08-01 00:00:00


In [4]:
# Compare regular_chip to childrens_medicaid_chip_group ratio over time
print(enrollment_by_rg[['month', 'regular_chip', 'childrens_medicaid_chip_group']].assign(
    ratio=lambda x: x['regular_chip'] / x['childrens_medicaid_chip_group']
).sort_values('month').tail(15))

KeyError: "['childrens_medicaid_chip_group'] not in index"

# Aggregate CHIP Enrollment Data (2014-2019)

In [5]:
chip_enrollment = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/chip-enrollment-detail.xlsx", sheet_name='CHIP Regular Caseload', skiprows=1)

# the last few rows are blank or contain notes, so we trim to just the data
chip_enrollment = chip_enrollment[0:138]

# standardize the column names to be lowercase, with underscores instead of spaces, and no special characters. This is necessary for loading into Snowflake.
chip_enrollment.columns = (chip_enrollment.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('*', '', regex=False)
    .str.replace('&', 'and', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('\u2019', '', regex=False)  # curly apostrophe
    .str.replace("'", '', regex=False)        # straight apostrophe fallback
    .str.replace('.', '_', regex=False)       # handles the .1 suffix
)

# add a timestamp column to track when the data was loaded into Snowflake
chip_enrollment['loaded_at'] = pd.Timestamp.now()

In [6]:
for col in chip_enrollment.columns:
    if col not in ['month', 'loaded_at']:
        has_decimal = (chip_enrollment[col] % 1 != 0).any()
        first_decimal = chip_enrollment[chip_enrollment[col] % 1 != 0]['month'].min() if has_decimal else None
        print(f"{col}: fractional={has_decimal}, first_occurrence={first_decimal}")

chip_caseload: fractional=False, first_occurrence=None
new_enrollment: fractional=False, first_occurrence=None
renewals: fractional=False, first_occurrence=None
disenrollment: fractional=False, first_occurrence=None


# Healthy Texas Women Enrollment (2014-2019)

In [7]:
hwt_caseload = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/healthy-texas-women-enrollment.xlsx", sheet_name='Summary')

# the last few rows are blank or contain notes, so we trim to just the data
hwt_caseload = hwt_caseload[0:138]

# standardize the column names to be lowercase, with underscores instead of spaces, and no special characters. This is necessary for loading into Snowflake.
hwt_caseload.columns = (hwt_caseload.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('*', '', regex=False)
    .str.replace('&', 'and', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('\u2019', '', regex=False)  # curly apostrophe
    .str.replace("'", '', regex=False)        # straight apostrophe fallback
    .str.replace('.', '_', regex=False)       # handles the .1 suffix
)

# rename the columns to be more descriptive and clear about the risk groups.
hwt_caseload = hwt_caseload.rename(columns={
    'healthy_texas_women_caseload':   'month',
    'unnamed:_1': 'caseload',
})

# add a timestamp column to track when the data was loaded into Snowflake
hwt_caseload['loaded_at'] = pd.Timestamp.now()

In [8]:
print(hwt_caseload.head())

                 month       caseload                  loaded_at
0  2026-02-01 00:00:00  357189.038319 2026-05-18 06:08:39.018063
1  2026-01-01 00:00:00  359137.461471 2026-05-18 06:08:39.018063
2  2025-12-01 00:00:00  365103.390303 2026-05-18 06:08:39.018063
3  2025-11-01 00:00:00  369671.509855 2026-05-18 06:08:39.018063
4  2025-10-01 00:00:00  377313.683691 2026-05-18 06:08:39.018063


In [9]:
print(len(enrollment_by_rg))       # should be 138
print(len(chip_enrollment))  # should be 138
print(len(hwt_caseload))   # should be 138

138
138
138


In [11]:
for col in hwt_caseload.columns:
    if col not in ['month', 'loaded_at']:
        has_decimal = (hwt_caseload[col] % 1 != 0).any()
        first_decimal = hwt_caseload[hwt_caseload[col] % 1 != 0]['month'].min() if has_decimal else None
        print(f"{col}: fractional={has_decimal}, first_occurrence={first_decimal}")

caseload: fractional=True, first_occurrence=2025-08-01 00:00:00


# Medicaid Timeliness Data 

In [14]:
apps = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/timeliness/timeliness-medicaid-jan-2025.xlsx", sheet_name='Medicaid', skiprows=3)

redets = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/timeliness/timeliness-medicaid-jan-2025.xlsx", sheet_name='Medicaid', skiprows=25)

In [15]:
apps = apps[1:19]  # trim first empty row and total row

redets = redets[1:19]  # trim first empty row and total row


In [16]:
print(len(apps))  
print(len(redets))

18
18


In [17]:
print(apps.head(3))
print(redets.head(3))

  Region Disposed Timely   Percent
1     01    22469  18434  0.820419
2  02/09    15149  11720  0.773648
3     03    48393  40082   0.82826
  Region  Disposed   Timely   Percent
1     01    9415.0   9384.0  0.996707
2  02/09    7012.0   7000.0  0.998289
3     03   22668.0  22641.0  0.998809


In [18]:
from pathlib import Path
import re

timeliness_folder = Path("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/timeliness/")

all_timeliness = []

for file in sorted(timeliness_folder.glob("*.xlsx")):
    # extract report_month from filename
    match = re.search(r'medicaid-(\w+-\d{4})\.xlsx', file.name)
    if not match:
        print(f"Skipping {file.name} - could not parse month")
        continue
    report_month = pd.to_datetime(match.group(1), format='mixed')
    
    # read applications table
    apps = pd.read_excel(file, sheet_name='Medicaid', skiprows=3)
    apps = apps[1:19]
    apps['record_type'] = 'applications'
    
    # read redeterminations table
    redets = pd.read_excel(file, sheet_name='Medicaid', skiprows=25)
    redets = redets[1:19]
    redets['record_type'] = 'redeterminations'
    
    # union both tables
    combined = pd.concat([apps, redets], ignore_index=True)
    
    # add report_month
    combined['report_month'] = report_month
    
    all_timeliness.append(combined)
    print(f"Processed {file.name}: {len(combined)} rows")

# concatenate all months
timeliness = pd.concat(all_timeliness, ignore_index=True)
print(f"\nTotal rows: {len(timeliness)}")

Processed timeliness-medicaid-april-2024.xlsx: 36 rows
Processed timeliness-medicaid-april-2025.xlsx: 36 rows
Processed timeliness-medicaid-aug-2024.xlsx: 36 rows
Processed timeliness-medicaid-aug-2025.xlsx: 36 rows
Processed timeliness-medicaid-dec-2024.xlsx: 36 rows
Processed timeliness-medicaid-dec-2025.xlsx: 36 rows
Processed timeliness-medicaid-feb-2024.xlsx: 36 rows
Processed timeliness-medicaid-feb-2025.xlsx: 36 rows
Processed timeliness-medicaid-jan-2024.xlsx: 36 rows
Processed timeliness-medicaid-jan-2025.xlsx: 36 rows
Processed timeliness-medicaid-july-2024.xlsx: 36 rows
Processed timeliness-medicaid-july-2025.xlsx: 36 rows
Processed timeliness-medicaid-june-2024.xlsx: 36 rows
Processed timeliness-medicaid-june-2025.xlsx: 36 rows
Processed timeliness-medicaid-march-2024.xlsx: 36 rows
Processed timeliness-medicaid-march-2025.xlsx: 36 rows
Processed timeliness-medicaid-may-2024.xlsx: 36 rows
Processed timeliness-medicaid-may-2025.xlsx: 36 rows
Processed timeliness-medicaid-nov-

In [19]:
print(timeliness.columns.tolist())
print(timeliness.head(3))

['Region', 'Disposed', 'Timely', 'Percent', 'record_type', 'report_month']
  Region Disposed Timely   Percent   record_type report_month
0     01    18192   6260  0.344107  applications   2024-04-01
1  02/09    19480  10561  0.542146  applications   2024-04-01
2     03    65717  29251  0.445106  applications   2024-04-01


In [20]:
print(timeliness['record_type'].value_counts())

record_type
applications        432
redeterminations    432
Name: count, dtype: int64


In [21]:
timeliness.columns = (timeliness.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

timeliness['loaded_at'] = pd.Timestamp.now()

geographic_regions = ['01', '02/09', '03', '04', '05', '06', '07', '08', '10', '11']
timeliness['is_geographic_region'] = timeliness['region'].isin(geographic_regions)

In [22]:
timeliness['is_geographic_region'].value_counts()

is_geographic_region
True     480
False    384
Name: count, dtype: int64

In [23]:
print(timeliness[timeliness['region'] == 'TOTAL'][['region', 'record_type', 'report_month']].head(5))
timeliness = timeliness[timeliness['region'] != 'TOTAL']
print(timeliness.columns)

   region       record_type report_month
17  TOTAL      applications   2024-04-01
35  TOTAL  redeterminations   2024-04-01
53  TOTAL      applications   2025-04-01
71  TOTAL  redeterminations   2025-04-01
89  TOTAL      applications   2024-08-01
Index(['region', 'disposed', 'timely', 'percent', 'record_type',
       'report_month', 'loaded_at', 'is_geographic_region'],
      dtype='object')


In [24]:
print(f"Row count: {len(timeliness)}")
print(f"Unique months: {timeliness['report_month'].nunique()}")
print(f"is_geographic_region dtype: {timeliness['is_geographic_region'].dtype}")
print(f"is_geographic_region values: {timeliness['is_geographic_region'].unique()}")

Row count: 816
Unique months: 24
is_geographic_region dtype: bool
is_geographic_region values: [ True False]


In [26]:
print(timeliness[timeliness['disposed'].isna()]['region'].value_counts())

region
MEPD         48
ST OFFICE    10
UNKNOWN       4
Name: count, dtype: int64


# Medicaid Enrollment by County 

In [2]:
county_enrollment = pd.read_excel('/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/county/medicaid-enrollment-by-county-final-april-2024.xlsx', sheet_name='Summary', skiprows=2)

county_enrollment = county_enrollment[0:255] # trim to just the data rows, excluding notes at the end

In [14]:
RAW = Path("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw")
COUNTY_DIR = RAW / "county"
all_frames = []

for filepath in sorted(COUNTY_DIR.glob("*.xlsx")):
    filename = filepath.name
    report_month = parse_report_month(filename)

    df = pd.read_excel(filepath, sheet_name='Summary', skiprows=2, header=0)

    df.columns = [
        'hhsc_county_code',
        'county',
        'medicaid_caseload',
        'aged_and_medicare_related',
        'disability_related',
        'parents',
        'pregnant_women',
        'breast_and_cervical_cancer',
        'childrens_medicaid',
        'medicaid_clients_under_21',
        'medicaid_clients_21_and_older'
    ]

    # drop total row and footnotes — keep only rows with a numeric county code
    df = df[pd.to_numeric(df['hhsc_county_code'], errors='coerce').notna()].copy()
    df['hhsc_county_code'] = df['hhsc_county_code'].astype(int)

    df['report_month'] = pd.to_datetime(report_month)
    df['loaded_at']    = pd.Timestamp.now()

    all_frames.append(df)
    print(f"  Processed {filename}: {len(df)} rows")

county_combined = pd.concat(all_frames, ignore_index=True)
print(f"  Total rows: {len(combined)}")

  Processed medicaid-enrollment-by-county-final-april-2024.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-april-2025.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-aug-2024.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-aug-2025.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-dec-2024.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-feb-2024.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-feb-2025.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-jan-2024.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-jan-2025.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-july-2024.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-july-2025.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-jun-2024.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-jun-2025.xlsx: 255 rows
  Processed medicaid-enrollment-by-county-final-mar-2024.x

In [16]:
print(county_combined.columns.tolist())
print(county_combined.dtypes)
print(county_combined.head(3))

['hhsc_county_code', 'county', 'medicaid_caseload', 'aged_and_medicare_related', 'disability_related', 'parents', 'pregnant_women', 'breast_and_cervical_cancer', 'childrens_medicaid', 'medicaid_clients_under_21', 'medicaid_clients_21_and_older', 'report_month', 'loaded_at']
hhsc_county_code                          int64
county                                   object
medicaid_caseload                       float64
aged_and_medicare_related               float64
disability_related                      float64
parents                                 float64
pregnant_women                          float64
breast_and_cervical_cancer              float64
childrens_medicaid                      float64
medicaid_clients_under_21               float64
medicaid_clients_21_and_older           float64
report_month                     datetime64[ns]
loaded_at                        datetime64[us]
dtype: object
   hhsc_county_code    county  medicaid_caseload  aged_and_medicare_related  \
0       

In [17]:
for col in county_combined.columns:
    if col not in ['hhsc_county_code', 'county', 'report_month', 'loaded_at']:
        has_decimal = (county_combined[col] % 1 != 0).any()
        first_decimal = county_combined[county_combined[col] % 1 != 0]['report_month'].min() if has_decimal else None
        print(f"{col}: fractional={has_decimal}, first_occurrence={first_decimal}")

medicaid_caseload: fractional=False, first_occurrence=None
aged_and_medicare_related: fractional=False, first_occurrence=None
disability_related: fractional=False, first_occurrence=None
parents: fractional=False, first_occurrence=None
pregnant_women: fractional=False, first_occurrence=None
breast_and_cervical_cancer: fractional=False, first_occurrence=None
childrens_medicaid: fractional=False, first_occurrence=None
medicaid_clients_under_21: fractional=False, first_occurrence=None
medicaid_clients_21_and_older: fractional=False, first_occurrence=None


# MCO Enrollment by SDA

In [35]:
mco = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/mco-enrollment-by-sda-final-sfy25.xlsx", sheet_name="Caseload_source", skiprows=1)

In [ ]:
mco = mco[0:203] # trim to just the data rows, excluding notes at the end

In [40]:
mco

,Unnamed: 0,Dallas,Tarrant,Harris,Nueces,Bexar,Travis,El Paso,Lubbock,Hildago,Jefferson,MRSA Central,MRSA North,MRSA West,EPO*,Total
0,Wellpoint / Amerigroup,231809.083333,110900.500000,7.289092e+04,6453.083333,11006.583333,0.000000,1375.500000,15156.666667,0.000000,17207.500000,16226.083333,62856.833333,42837.000000,0.00,5.887198e+05
1,CHIP,14979.750000,6589.500000,3.215417e+03,0.000000,793.416667,0.000000,0.000000,0.000000,0.000000,267.000000,0.000000,0.000000,0.000000,0.00,2.584508e+04
2,Regular,13099.250000,5747.833333,2.143083e+03,NaN,510.250000,NaN,NaN,NaN,NaN,183.750000,NaN,NaN,NaN,NaN,2.168417e+04
3,Perinatal,1880.500000,841.666667,1.072333e+03,NaN,283.166667,NaN,NaN,NaN,NaN,83.250000,NaN,NaN,NaN,NaN,4.160917e+03
4,MEDICAID,216829.333333,104311.000000,6.967550e+04,6453.083333,10213.166667,0.000000,1375.500000,15156.666667,0.000000,16940.500000,16226.083333,62856.833333,42837.000000,0.00,5.628747e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,Total CHIP****,32390.250000,22244.250000,5.155983e+04,4098.000000,15584.666667,12531.250000,6822.500000,4258.333333,0.000000,4415.833333,0.000000,0.000000,0.000000,42990.25,1.968952e+05
212,Total Caseload,560158.666667,400142.250000,1.042243e+06,123366.250000,382125.250000,220897.333333,154447.583333,105367.500000,440208.000000,131026.666667,197273.333333,248747.250000,218263.500000,47186.00,4.271452e+06
213,Percent Managed Care,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214,Medicaid,0.939705,0.932549,9.456283e-01,0.952469,0.943171,0.927014,0.948312,0.935540,0.963438,0.947463,0.942474,0.942639,0.938193,0.00,9.427961e-01


In [41]:
import pandas as pd

f = "/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/mco-enrollment-by-sda-final-sfy25.xlsx"

df = pd.read_excel(f, sheet_name=0, header=None, nrows=25)
print(df.to_string())

                        0              1              2             3             4              5              6             7             8              9            10            11             12            13       14             15
0   State Fiscal Year 2025            NaN            NaN           NaN           NaN            NaN            NaN           NaN           NaN            NaN          NaN           NaN            NaN           NaN      NaN            NaN
1                      NaN         Dallas        Tarrant        Harris        Nueces          Bexar         Travis       El Paso       Lubbock        Hildago    Jefferson  MRSA Central     MRSA North     MRSA West     EPO*          Total
2   Wellpoint / Amerigroup  231809.083333       110900.5  72890.916667   6453.083333   11006.583333              0        1375.5  15156.666667              0      17207.5  16226.083333   62856.833333         42837        0      588719.75
3                     CHIP       14979.75       

In [18]:
df = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/mco-enrollment-by-sda-final-sfy25.xlsx", sheet_name=0, header=None)

# row 1 contains the SDA/column names; drop the last column (Total)
sda_columns = df.iloc[1, 1:-1].tolist()

# MCO blocks start at row 2, each block is exactly 10 rows
# rows 0-201 are data; row 202 onward is statewide totals — drop those
MCO_BLOCK_SIZE = 10
DATA_START_ROW = 2
DATA_END_ROW   = 202  # exclusive

# sub-program labels in fixed order within each MCO block
# row 0 = MCO total, row 1 = CHIP total, row 2 = Regular, row 3 = Perinatal
# row 4 = MEDICAID total, row 5 = STAR, row 6 = STAR+Plus, row 7 = Dual Demo
# row 8 = STAR Health, row 9 = STAR Kids
PROGRAM_MAP = {
    0: ('TOTAL',    'TOTAL'),
    1: ('CHIP',     'TOTAL'),
    2: ('CHIP',     'Regular'),
    3: ('CHIP',     'Perinatal'),
    4: ('MEDICAID', 'TOTAL'),
    5: ('MEDICAID', 'STAR'),
    6: ('MEDICAID', 'STAR+Plus'),
    7: ('MEDICAID', 'Dual Demo'),
    8: ('MEDICAID', 'STAR Health'),
    9: ('MEDICAID', 'STAR Kids'),
}

records = []

for block_start in range(DATA_START_ROW, DATA_END_ROW, MCO_BLOCK_SIZE):
    mco_name = df.iloc[block_start, 0]

    for offset, (program, sub_program) in PROGRAM_MAP.items():
        row = df.iloc[block_start + offset, 1:-1]  # drop first col (label) and last col (total)

        for sda, value in zip(sda_columns, row):
            # NaN means the MCO does not serve that SDA — skip entirely
            if pd.isna(value):
                continue

            records.append({
                'mco_name':        mco_name,
                'program':         program,
                'sub_program':     sub_program,
                'sda':             sda,
                'enrollment':      value,
                'enrollment_type': 'sfy_monthly_average',
                'fiscal_year':     2025,
                'source_file':     'mco-enrollment-by-sda-final-sfy25.xlsx',
                'loaded_at':       pd.Timestamp.now()
            })

mco_sda = pd.DataFrame(records)

# clean the mco_name column — one MCO has a newline character in its name
mco_sda['mco_name'] = mco_sda['mco_name'].str.replace('\n', ' ', regex=False).str.strip()

print(f"Total rows: {len(mco_sda)}")
print(f"MCOs: {sorted(mco_sda['mco_name'].unique())}")
print(f"SDAs: {sorted(mco_sda['sda'].unique())}")
print(mco_sda.head(10))

Total rows: 1004
MCOs: ['Aetna', 'Blue Cross and Blue Shield of Texas', 'CHRISTUS', "Children's Medical Center", 'Cigna/Texas HealthSpring', 'Community First', 'Community Health Choice', "Cook Children's", 'Dell Children', 'Driscoll', 'El Paso First', 'FirstCare', 'Molina Healthcare of Texas', 'Parkland Community', 'Scott & White', 'Sendero', 'Superior', "Texas Children's", 'UnitedHealthcare/ Evercare of Texas', 'Wellpoint / Amerigroup']
SDAs: ['Bexar', 'Dallas', 'EPO*', 'El Paso', 'Harris', 'Hildago', 'Jefferson', 'Lubbock', 'MRSA Central', 'MRSA North', 'MRSA West', 'Nueces', 'Tarrant', 'Travis']
                 mco_name program sub_program        sda     enrollment  \
0  Wellpoint / Amerigroup   TOTAL       TOTAL     Dallas  231809.083333   
1  Wellpoint / Amerigroup   TOTAL       TOTAL    Tarrant  110900.500000   
2  Wellpoint / Amerigroup   TOTAL       TOTAL     Harris   72890.916667   
3  Wellpoint / Amerigroup   TOTAL       TOTAL     Nueces    6453.083333   
4  Wellpoint / Amer

In [ ]:
print(mco_sda.columns.tolist())
print(mco_sda.dtypes)
print(mco_sda.head(3))

['mco_name', 'program', 'sub_program', 'sda', 'enrollment', 'enrollment_type', 'fiscal_year', 'source_file', 'loaded_at']
mco_name                   object
program                    object
sub_program                object
sda                        object
enrollment                float64
enrollment_type            object
fiscal_year                 int64
source_file                object
loaded_at          datetime64[ns]
dtype: object
                 mco_name program sub_program      sda     enrollment  \
0  Wellpoint / Amerigroup   TOTAL       TOTAL   Dallas  231809.083333   
1  Wellpoint / Amerigroup   TOTAL       TOTAL  Tarrant  110900.500000   
2  Wellpoint / Amerigroup   TOTAL       TOTAL   Harris   72890.916667   

       enrollment_type  fiscal_year                             source_file  \
0  sfy_monthly_average         2025  mco-enrollment-by-sda-final-sfy25.xlsx   
1  sfy_monthly_average         2025  mco-enrollment-by-sda-final-sfy25.xlsx   
2  sfy_monthly_average      

In [20]:
print(mco_sda['program'].unique())
print(mco_sda['sub_program'].unique())

['TOTAL' 'CHIP' 'MEDICAID']
['TOTAL' 'Regular' 'Perinatal' 'STAR' 'STAR+Plus' 'Dual Demo' 'STAR Kids'
 'STAR Health']


In [23]:
print(mco_sda.groupby(['program', 'sub_program']).size().reset_index(name='count'))

    program  sub_program  count
0      CHIP    Perinatal     34
1      CHIP      Regular     34
2      CHIP        TOTAL    253
3  MEDICAID    Dual Demo     13
4  MEDICAID         STAR     45
5  MEDICAID  STAR Health     13
6  MEDICAID    STAR Kids     28
7  MEDICAID    STAR+Plus     38
8  MEDICAID        TOTAL    266
9     TOTAL        TOTAL    280


In [ ]:
mco_sda[mco_sda['program'] == 'TOTAL'].shape[0]     
mco_sda[mco_sda['sub_program'] == 'TOTAL'].shape[0]

799

In [28]:
mco_sda = mco_sda[~((mco_sda['program'] == 'TOTAL') | (mco_sda['sub_program'] == 'TOTAL'))]
print(len(mco_sda))
print(mco_sda.groupby(['program', 'sub_program']).size().reset_index(name='count'))

205
    program  sub_program  count
0      CHIP    Perinatal     34
1      CHIP      Regular     34
2  MEDICAID    Dual Demo     13
3  MEDICAID         STAR     45
4  MEDICAID  STAR Health     13
5  MEDICAID    STAR Kids     28
6  MEDICAID    STAR+Plus     38
